# Stage 5: Train the regression models (LIAR, 3 checkpoints)

**Stage:** 5 (training) · **Runs on:** Kaggle and locally (CPU)

Predicts LIAR's truthfulness score: 0 pants-fire, 1 false, 2 barely-true, 3 half-true,
4 mostly-true, 5 true. Only LIAR has these graded ratings.

| Model | Role |
|---|---|
| Dummy (mean) | always predicts the training average: the baseline every model must beat |
| Ridge | linear regression on TF-IDF, regularisation tuned with 5-fold cross-validation |
| LightGBM regressor | boosted trees on the dense vectors |

| | |
|---|---|
| Input | `liar_reg_*` split files, Stage 4 `liar_reg` vectors |
| Output | `dummy_reg__liar.joblib`, `ridge_reg__liar.joblib`, `lightgbm_reg__liar.joblib`, and `pred__<model>__liar.csv` for each |

**On Kaggle:** attach the `misinfo_utils` utility script and add these notebooks as inputs (**Add Input → Your Work → Notebooks**): `02_split` and `04_features`.

In [4]:
# Setup: import the shared code. Works on Kaggle and locally.
import sys
from pathlib import Path

# Locally, misinfo_utils.py is in <project>/src/. On Kaggle it is an attached Utility Script,
# so this loop finds nothing and the import below still works.
for folder in [Path.cwd(), *Path.cwd().parents]:
    if (folder / "src" / "misinfo_utils.py").exists():
        sys.path.insert(0, str(folder / "src"))
        break

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import misinfo_utils as mu

sns.set_theme(style="whitegrid")
pd.set_option("display.max_colwidth", 100)
paths = mu.setup("05")

misinfo_utils | stage 05 | running on local
  read from : D:\misinfo-project\data\raw, D:\misinfo-project\outputs
  write to  : D:\misinfo-project\outputs\05_models


In [5]:
import joblib
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV, KFold
from lightgbm import LGBMRegressor

RIDGE_GRID = {"alpha": [0.3, 1, 3, 10, 30]}

y_train = mu.load_split("liar_reg", "train")["score"].astype(float).values
X = {(kind, s): (mu.load_tfidf("liar_reg", s) if kind == "sparse" else mu.load_dense("liar_reg", s))
     for kind in ("sparse", "dense") for s in mu.SPLITS}

search = GridSearchCV(Ridge(), RIDGE_GRID, scoring="neg_mean_absolute_error",
                      cv=KFold(n_splits=5, shuffle=True, random_state=mu.SEED)).fit(X[("sparse", "train")], y_train)
print("Ridge best alpha:", search.best_params_["alpha"], "| cross-validated MAE:", round(-search.best_score_, 3))

models = {
    "dummy_reg": DummyRegressor(strategy="mean").fit(X[("sparse", "train")], y_train),
    "ridge_reg": search.best_estimator_,
    "lightgbm_reg": LGBMRegressor(n_estimators=400, num_leaves=31, learning_rate=0.05, min_child_samples=20,
                                  subsample=0.9, subsample_freq=1, colsample_bytree=0.8,
                                  random_state=mu.SEED, n_jobs=-1, verbose=-1).fit(X[("dense", "train")], y_train),
}

Ridge best alpha: 3 | cross-validated MAE: 1.257


In [6]:
val_rows = []
for name, model in models.items():
    joblib.dump(model, mu.output_path(mu.checkpoint_file(name, "liar"), subfolder="regression/liar"))
    kind = "sparse" if name in mu.SPARSE_MODELS else "dense"
    parts = []
    for s in ("val", "test"):
        df = mu.load_split("liar_reg", s)
        pred = np.clip(model.predict(X[(kind, s)]), 0, 5)
        parts.append(pd.DataFrame({"row_id": df["row_id"], "split": s, "pred_score": pred}))
        if s == "val":
            val_rows.append({"model": mu.MODEL_NAMES[name], **mu.reg_scores(df["score"], pred)})
    pd.concat(parts).to_csv(mu.output_path(mu.pred_file(name, "liar"), subfolder="regression/liar"), index=False)

pd.DataFrame(val_rows).set_index("model").round(3)   # validation only; test is scored in Stage 7

,MAE,RMSE,R2,within_1_level_%
model,,,,
Dummy (mean),1.348,1.561,-0.000,37.018
Ridge,1.268,1.495,0.083,41.971
LightGBM regressor,1.290,1.521,0.051,40.667


**Reading it:** MAE is the average error in rating levels (e.g. 1.2 = off by about one level).
A model is only useful if it clearly beats the Dummy baseline.